In [7]:
import json
import copy
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
import torch
import sys
import os
from fuzzywuzzy import fuzz
ROOT_DIR = "../.."
sys.path.append(f'{ROOT_DIR}/src')

from prompts.category_mapping import category_mapping_cardiac, category_mapping_cholec, category_mapping_massmaps, category_mapping_sepsis, category_mapping_supernova
# from cardiac import make_alignment_matrix as cardiac_make_alignment_matrix
# from sepsis import make_alignment_matrix as sepsis_make_alignment_matrix
# from supernova import make_alignment_matrix as supernova_make_alignment_matrix

def make_alignment_matrix(categories, claims, claims_by_category, category_alignment_scores):
    """
    Args:
        categories (list[str]): A list of all expertcategories.
        claims (list[str]): A list of all atomic claims.
        claims_by_category (dict[str, list[str]]): A dictionary where the keys are the categories and the values are lists of claims that are aligned with the category.
        category_alignment_scores (dict[str, float]): A dictionary where the keys are the categories and the values are the alignment scores.
    Returns:
        list[list[float]]: A matrix of alignment scores for the claims in the categories.
    """
    matrix = np.zeros((len(claims), len(categories)))
    for i, claim in enumerate(claims):
        for j, category in enumerate(categories):
            if any(fuzz.ratio(claim, c) > 90 for c in claims_by_category[category]):
                matrix[i, j] = category_alignment_scores[category]
    return matrix

# eval_model_name = 'gpt-5-mini-2025-08-07'
models = [
    "gpt-5.2-pro-2025-12-11",
    "gpt-5-mini-2025-08-07",
    "claude-opus-4-5-20251101",
    "claude-haiku-4-5-20251001",
    "gemini-2.5-pro",
    "gemini-2.5-flash"
]

methods = [
    'vanilla', 
    'cot', 
    'socratic', 
    'subq'
]

for dataset_name in [
    "massmaps", "cholec"
    # "cardiac", "sepsis", "supernova",
]:
    print(f"=== Using dataset_name {dataset_name} ===")
    if dataset_name == "cardiac":
        categories_list = [name for name, _ in sorted(category_mapping_cardiac["name2id"].items(), key=lambda x: x[1])]
    elif dataset_name == "sepsis":
        categories_list = [name for name, _ in sorted(category_mapping_sepsis["name2id"].items(), key=lambda x: x[1])]
    elif dataset_name == "supernova":
        categories_list = [name for name, _ in sorted(category_mapping_supernova["name2id"].items(), key=lambda x: x[1])]
    elif dataset_name == "massmaps":
        categories_list = [name for name, _ in sorted(category_mapping_massmaps["name2id"].items(), key=lambda x: x[1])]
    elif dataset_name == "cholec":
        categories_list = [name for name, _ in sorted(category_mapping_cholec["name2id"].items(), key=lambda x: x[1])]
    else:
        raise ValueError(f"Unknown dataset {dataset_name}")
    # for eval_model_name in ['gpt-5-mini-2025-08-07', 'gemini-2.5-flash-lite', 'qwen2.5-vl']:
        # print(f"=== Using eval_model_name {eval_model_name} ===")
    for model in models:
        print(f"=== Using model {model} ===")
        for method in methods:
            print(f"=== Using method {method} ===")

            # load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}.json')
            load_path = os.path.join(ROOT_DIR, f'results/{method}/{dataset_name}_{model}.json')
            save_path2 = os.path.join(ROOT_DIR, f'results/{method}/{dataset_name}_{model}.2.json')

            if not Path(load_path).exists():
                print(load_path, " doesn't exist")
                continue

            with open(load_path) as input_file:
                results = json.load(input_file)

            new_results = []

            num_examples = len(results)
            for di in tqdm(range(num_examples)):
                result = results[di]

                if 'all_claims' in result:
                    all_claims = result['all_claims']
                else:
                    all_claims = result['claims']

                alignment_matrix = make_alignment_matrix(
                    categories_list,
                    all_claims,
                    result['claims_by_category'],
                    result['category_alignment_scores']
                )

                final_alignment_score = alignment_matrix.max(axis=-1).mean()
                if np.isnan(final_alignment_score):
                    print(f'example {idx} final_alignment_score is NaN')
                result['final_alignment_score'] = final_alignment_score

                # save
                save_dict = {}
                for k, v in result.items():
                    if not isinstance(v, torch.Tensor):
                        save_dict[k] = v # if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
                # with open(save_path, 'wt') as output_file:
                #     json.dump(save_dict, output_file)

                new_results.append(save_dict)


            with open(save_path2, 'wt') as output_file:
                json.dump(new_results, output_file, indent=4)

=== Using dataset_name massmaps ===
=== Using model gpt-5.2-pro-2025-12-11 ===
=== Using method vanilla ===


  0%|          | 0/100 [00:00<?, ?it/s]

=== Using method cot ===


  0%|          | 0/100 [00:00<?, ?it/s]

=== Using method socratic ===


...

  0%|          | 0/100 [00:00<?, ?it/s]

=== Using method socratic ===


  0%|          | 0/100 [00:00<?, ?it/s]

=== Using method subq ===


  0%|          | 0/100 [00:00<?, ?it/s]